In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import torch
print("GPU:",torch.cuda.get_device(0)if torch.cuda.is_available() else "yok (CPU ile çalışıyor)")

GPU: yok (CPU ile çalışıyor)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

Model_adı = "Qwen/Qwen2.5-1.5B-Instruct"
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(Model_adı, trust_remote_code=True)
print(f"Kullnaılan Cihaz: {cihaz}")
model = AutoModelForCausalLM.from_pretrained(Model_adı, trust_remote_code=True).to(cihaz)
model.eval()
print(f"parametre sayısı:")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

parametre sayısı:


In [ ]:
ornek = "Yapay zeka dünyayı değiştiriyor."
tokenler = tokenizer.tokenize(ornek)
idler = tokenizer.encode(ornek)
print(f"Tokenler: {tokenler}")
print(f"Idler: {idler}")
print(f"geri çözülmüş:",tokenizer.decode(idler))


Tokenler: ['Y', 'ap', 'ay', 'Ġz', 'eka', 'ĠdÃ¼ny', 'ayÄ±', 'ĠdeÄŁiÅŁtir', 'iyor', '.']
Idler: [56, 391, 352, 1147, 52313, 139704, 126379, 133382, 83342, 13]
geri çözülmüş: Yapay zeka dünyayı değiştiriyor.


In [ ]:
def sor(soru,sistem= " Sen yardımsever bir asistansın. Türkçe ve kısa cevap ver.", max_yeni_token = 200, temperature =0.7,top_p=0.9):
  mesajlar =[
      {"role":"system","content":sistem},
      {"role":"user","content":soru},
  ]

  metin = tokenizer.apply_chat_template(mesajlar,tokenize=False,add_generation_prompt=True)
  tokenler = tokenizer(metin,return_tensors = "pt").to(cihaz)
  with torch.no_grad():
    cikti = model.generate(
      **tokenler,
      max_new_tokens= max_yeni_token,
      do_sample = temperature >0,
      temperature= temperature if temperature >0 else None,
      top_p = top_p if temperature >0  else None,
      pad_token_id = tokenizer.eos_token_id,
    )
  yeni = cikti[0][tokenler["input_ids"].shape[1]:]
  return tokenizer.decode(yeni,skip_special_tokens=True)

  print(sor("Sevgiliye her hafta çicek alınır mı ?"))

In [ ]:
sorular = [
    "Fatih 3'ü bir ada sever mi ?",
    "Komşular iyi midir ?",
    "Maria mı haklıydı yoksa Ragıp Efendi mi ?",
    "Evlenmeden önce kaç yıl sevgili kalınmalı"

  ]
for s in sorular:
  print(f"❓ {s}\n 💔 sor{sor(s)}\n"+"--"*4)

❓ Fatih 3'ü bir ada sever mi ?
 💔 sorSınavlar, Fatih 3'in bir ada olduğunu düşünürsünüz? Ancak, Fatih 3'nin yerini belirlemek için tam olarak bilgi ederim. Size nasıl yardımcı olabilirim?
--------
❓ Komşular iyi midir ?
 💔 sorKomşular, herhangi bir sağlık sorununu veya güvenliği için gereklidir. Ancak, genellikle genel sağlık sorunlarını karşılamaktadır. Özellikle sağlık durumlarına olanak tanımalarını da uygun olmakla beraber.

Ancak, her bir komşu ne kadar iyisini de biliyor musun? Komşular, aynı zamanda diğer bir ülke veya bölge gibi başka bir küresel aracılığıyla farklı durumlardan yararlanabilirler. Bu durumda, özellikle sağlık sorunları hakkında daha fazla bilgi alması önemlidir.
--------
❓ Maria mı haklıydı yoksa Ragıp Efendi mi ?
 💔 sorRagıp Efendi yoktu, Maria haklıydı.
--------


KeyboardInterrupt: 

In [ ]:
soru ="Aşk ve  kasımpatı ile ilgili bir şiir yaz"
for t in [0,0.7,1.3]:
  print(f"temperature={t}:{sor(soru,temperature=t,max_yeni_token=60)}\n")

In [ ]:
class Sohbet:
  def __init__(self,sistem ="Sen yardımsever bir asistan.Türkçe cevap ver"):

    self.gecmis=[{"role":"system","content":sistem}]

  def mesaj(self,kullanici_mesaji,max_yeni_token=200):
    self.gecmis.append({"role":"user","content":kullanici_mesaji})
    metin = tokenizer.apply_chat_template(self.gecmis,tokenize=False,add_generation_prompt=True)
    tokenler = tokenizer(metin,return_tensors = "pt").to(cihaz)
    with torch.no_grad():
      cikti = model.generate(**tokenler,max_new_tokens=max_yeni_token,do_sample=True,temperature =0.2,top_p=0.9,
      pad_token_id = tokenizer.eos_token_id)
    cevap = tokenizer.decode(cikti[0][tokenler["input_ids"].shape[1]:],skip_special_tokens=True)
    self.gecmis.append({"role":"assistant","content":cevap})
    return cevap

sohbet = Sohbet()
print("🤖",sohbet.mesaj("Merhaba , Benim adım ilgisu ve ben 2 kedi annesiyim"))
print("🤖",sohbet.mesaj("Adımı ve ne annesi olduğumu hatırlıyor musun ?"))

🤖 Merhaba, ilgisu! Ben de çok sevgilim. Kediler çok güzel ve heyecan verici. Size nasıl yardımcı olabilirim?
🤖 Evet, size her zaman yardımcı olacağım. Adınız "Ilgisu" ve anneniz "Kediler" olarak bilinen kişi miydi? Size nasıl yardımcı olabileceğimi anlayarak yardımcı olmak için bu bilgileri bana verebilir misiniz?
